# The wolf detector that only saw snow

A model that scores well on its test set and has learned the wrong thing. We
build one on purpose, with two features: *ear shape* (the real signal) and
*snow in the background* (the shortcut).

In [1]:
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)

def make_data(n, p_snow_given_wolf, p_snow_given_husky):
    is_wolf = rng.random(n) < 0.5
    ears = np.where(is_wolf, rng.normal(0.35, 0.6, n), rng.normal(-0.35, 0.6, n))  # weak but real signal
    snow = np.where(is_wolf, rng.random(n) < p_snow_given_wolf, rng.random(n) < p_snow_given_husky)
    return np.column_stack([ears, snow.astype(float)]), is_wolf.astype(int)

X_train, y_train = make_data(400, p_snow_given_wolf=0.95, p_snow_given_husky=0.05)   # snow ~ label
X_test,  y_test  = make_data(200, p_snow_given_wolf=0.95, p_snow_given_husky=0.05)   # same bias!
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", model.score(X_test, y_test))
print("coefficients  ears: %.2f   snow: %.2f" % tuple(model.coef_[0]))

test accuracy: 0.97
coefficients  ears: 1.56   snow: 4.69


Excellent test score, and the coefficient on *snow* is larger than the one on
*ears*: the model leaned on the background.

## The world where the model is used

In [2]:
X_zoo, y_zoo = make_data(200, p_snow_given_wolf=0.5, p_snow_given_husky=0.5)   # snow no longer tells you anything
print("accuracy at the zoo:", model.score(X_zoo, y_zoo))

accuracy at the zoo: 0.525


## Ask the feature, not the score

A model trained where the shortcut does not exist has to use the real signal.
It does worse on the biased test set while being right for the right reasons.

In [3]:
X_fair, y_fair = make_data(400, p_snow_given_wolf=0.5, p_snow_given_husky=0.5)
fair = LogisticRegression().fit(X_fair, y_fair)
print("coefficients  ears: %.2f   snow: %.2f" % tuple(fair.coef_[0]))
print("fair model on the biased test set:", fair.score(X_test, y_test), "   at the zoo:", fair.score(X_zoo, y_zoo))

coefficients  ears: 1.86   snow: -0.53
fair model on the biased test set: 0.57    at the zoo: 0.735


**What we used:** features, generalization, why a test set can share the
bias of the training set, and explanation (session 10, sections 4, 6, 8 and 12).